In [22]:
import polars as pl
import os
import numpy as np

In [4]:
INTERVALS = '/scratch/tsoies-Expression/intervals/'

In [ ]:
for split_method in [split_method for split_method in os.listdir(INTERVALS) if os.path.isdir(os.path.join(INTERVALS, split_method))]:
    print(split_method)
    

random_enformer
intervals
cactus_random
cactus_borzoi
random_borzoi
blastp_borzoi
blastp_enformer
cactus_enformer
random
blastp_random


In [50]:
def calculate_similarity(split_root:str,
                            splitName1:str,
                            splitName2:str,
                            ):
    genesDirSplit1 = os.path.join(split_root, splitName1, 'genes')
    genesDirSplit2 = os.path.join(split_root, splitName2, 'genes')
    for genome in os.listdir(genesDirSplit1):
        train_valid_table_split_1 = pl.read_csv(os.path.join(genesDirSplit1, genome, f'{genome}_genes.tsv'), separator='\t')
        train_valid_table_split_2 = pl.read_csv(os.path.join(genesDirSplit2, genome, f'{genome}_genes.tsv'), separator='\t')
        joined_df = train_valid_table_split_1.join(train_valid_table_split_2, on = 'gene_id')
        joined_df = joined_df.with_columns(equal = (pl.col('split') == pl.col('split_right')))
        similarity = joined_df['equal'].sum() / len(joined_df)
        print(f"{genome}: {similarity}")

In [69]:
def calculate_splits_ratio(split_root:str,
                            splits:list|None = None
                            ):
    if splits is None:
        return None
    
    genesDir = os.path.join(split_root, splits[0], 'genes')
    for genome in os.listdir(genesDir):
        for split in splits:
            genesDir = os.path.join(split_root, split, 'genes')
            train_valid_table = pl.read_csv(os.path.join(genesDir, genome, f'{genome}_genes.tsv'), separator='\t')
            train_count = train_valid_table.filter(pl.col('split') == 'train').shape[0]
            valid_count = train_valid_table.filter(pl.col('split') == 'valid').shape[0]

            print(f'split: {split} | genome: {genome} | train: {train_count} | test: {valid_count}')
            
        print('--------------------------------------------------------------------------------')
    

In [71]:
calculate_splits_ratio(split_root=INTERVALS, splits=['cactus_enformer', 'random_enformer', 'blastp_enformer'])

split: cactus_enformer | genome: GCF_002263795.3 | train: 23354 | test: 2505
split: random_enformer | genome: GCF_002263795.3 | train: 21724 | test: 3650
split: blastp_enformer | genome: GCF_002263795.3 | train: 18394 | test: 5542
--------------------------------------------------------------------------------
split: cactus_enformer | genome: GCF_036323735.1 | train: 28401 | test: 2856
split: random_enformer | genome: GCF_036323735.1 | train: 26043 | test: 4335
split: blastp_enformer | genome: GCF_036323735.1 | train: 23182 | test: 5959
--------------------------------------------------------------------------------
split: cactus_enformer | genome: GCF_011100685.1 | train: 27137 | test: 2792
split: random_enformer | genome: GCF_011100685.1 | train: 24891 | test: 4209
split: blastp_enformer | genome: GCF_011100685.1 | train: 21773 | test: 6021
--------------------------------------------------------------------------------
split: cactus_enformer | genome: GCF_000003025.6 | train: 18238 

In [51]:
calculate_similarity(split_root=INTERVALS, splitName1='cactus_enformer', splitName2='random_enformer')

GCF_002263795.3: 0.6362688900468995
GCF_036323735.1: 0.6371451650126932
GCF_011100685.1: 0.6420679843811484
GCF_000003025.6: 0.6275386955354746
GCF_000001405.40: 1.0
GCF_041296265.1: 0.6397892640303254
GCF_016772045.2: 0.6244778914014115
GCF_000001635.27: 0.6369976842392044
GCF_964237555.1: 0.6390124159521862
GCF_049350105.2: 0.6257495802350683


In [52]:
a = calculate_similarity(split_root=INTERVALS, splitName1='blastp_enformer', splitName2='random_enformer')

GCF_002263795.3: 0.5250303977766198
GCF_036323735.1: 0.5327429033002539
GCF_011100685.1: 0.5429064382359173
GCF_000003025.6: 0.509841639080254
GCF_000001405.40: 0.832257883514757
GCF_041296265.1: 0.5200616788203926
GCF_016772045.2: 0.5196960967881319
GCF_000001635.27: 0.5370113063615312
GCF_964237555.1: 0.5297947276672952
GCF_049350105.2: 0.5282741664667786


In [40]:
calculate_similarity(split_root=INTERVALS, splitName1='cactus_enformer', splitName2='blastp_enformer')

GCF_002263795.3: 0.7347924266110821
GCF_036323735.1: 0.7399896145857373
GCF_011100685.1: 0.7501286436419772
GCF_000003025.6: 0.7493960812382572
GCF_000001405.40: 0.832257883514757
GCF_041296265.1: 0.7444505123839507
GCF_016772045.2: 0.7357050266455423
GCF_000001635.27: 0.7430050401852608
GCF_964237555.1: 0.7356362730797965
GCF_049350105.2: 0.7465219477092828


In [48]:
calculate_similarity(split_root=INTERVALS, splitName1='random', splitName2='random_borzoi')

GCF_002263795.3: 0.60055584505819
GCF_036323735.1: 0.5955746595891992
GCF_011100685.1: 0.6004479825650029
GCF_000003025.6: 0.6074080701440459
GCF_000001405.40: 0.6115357318338156


gene_id,strand,chromosome,TSS,TES,gene_name,split,strand_right,chromosome_right,TSS_right,TES_right,gene_name_right,split_right,equal
str,str,str,i64,i64,str,str,str,str,i64,i64,str,str,bool
"""LOC100288069""","""-""","""NC_000001.11""",778634,725759,"""LOC100288069""","""train""","""-""","""NC_000001.11""",778634,725759,"""LOC100288069""","""train""",true
"""LINC01409""","""+""","""NC_000001.11""",778798,810065,"""LINC01409""","""train""","""+""","""NC_000001.11""",778798,810065,"""LINC01409""","""train""",true
"""LOC124903817""","""+""","""NC_000001.11""",791598,799081,"""LOC124903817""","""train""","""+""","""NC_000001.11""",791598,799081,"""LOC124903817""","""train""",true
"""FAM87B""","""+""","""NC_000001.11""",817371,819834,"""FAM87B""","""train""","""+""","""NC_000001.11""",817371,819834,"""FAM87B""","""train""",true
"""LINC00115""","""-""","""NC_000001.11""",827522,826206,"""LINC00115""","""train""","""-""","""NC_000001.11""",827522,826206,"""LINC00115""","""train""",true
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""H2AB3""","""-""","""NC_000023.11""",155460005,155459415,"""H2AB3""","""train""","""-""","""NC_000023.11""",155460005,155459415,"""H2AB3""","""train""",true
"""TMLHE-AS1""","""+""","""NC_000023.11""",155466540,155494110,"""TMLHE-AS1""","""train""","""+""","""NC_000023.11""",155466540,155494110,"""TMLHE-AS1""","""train""",true
"""TMLHE""","""-""","""NC_000023.11""",155612952,155489011,"""TMLHE""","""train""","""-""","""NC_000023.11""",155612952,155489011,"""TMLHE""","""train""",true
